# Notebook 25: fine-tuning Atom-JEPA on the 18-target table

Atom-JEPA (Petersen, Tirsgaard, Cornet, Jordahn and Schmidt, DTU, arXiv 2610.08400, 6 Oct 2026)
is an EquiformerV3 3D network pretrained, with no labels, on about 19 million Uni-Mol
molecules by predicting the latent representation of one part of a molecule from another. Code
`github.com/khelverskovp/atom-jepa` (MIT), weights `huggingface.co/atom-jepa/atom-jepa`
(CC-BY-4.0), installed here as the `atom-jepa` package. **Cite them in the method report.**

**Why it is worth a GPU run.** Every base model we have is either a chemprop network or TabPFN
on a chemprop-family embedding, and every arm we add correlates above 0.9 with one already in
the blend. The paper reports that fine-tuned Atom-JEPA was best or statistically tied on 9 of 9
endpoints of ExpansionRx (the previous OpenADMET challenge), that pretraining cut its error
8.5% against the same network from scratch, and that its errors are among the least correlated
with other model families. Those are the authors' claims on ADMET data, not CYP data, and the
network is also the only 3D model in their comparison, so the gain could come from
equivariance and conformer averaging and not from the pretraining. It has never been tried on
this challenge here. Frozen Uni-Mol embeddings (nb17) added nothing; this is full fine-tuning.

**What this notebook does, following their ADMET recipe.**

| | |
| :--- | :--- |
| targets | the 18 heads of nb21: 4 challenge pIC50, 4 screen log2FC, ChEMBL IC50 for 5 isoforms, AID 1851 `Max_Response` for 5 isoforms; scored heads weight 1.0, the rest 0.3 |
| input | 10 RDKit ETKDGv3 plus MMFF94 conformers per challenge compound (3 per public compound), hydrogens included, largest fragment only |
| training | one random conformer per molecule per epoch; whole encoder fine-tuned; AdamW, encoder lr 5e-5, head lr 5e-4, cosine decay with linear warmup, Huber loss on standardised targets, weight EMA 0.99, gradient clip 100, L2-SP 1e-5 toward the pretrained weights |
| head | atom-level l=0 scalars and per-degree norms, mean-pooled (768 numbers), a shared trunk `768 to 256` with dropout 0.1, then one `256 to 64 to 1` head per target |
| prediction | **the mean over a compound's 10 conformers**, then the mean over the five fold models for the holdout and the 750 blind compounds |

**Differences from the paper, forced by Colab and said plainly.** Public compounds enter as a
random pool of `PUBLIC_POOL` (6,000 of the 28,912) with `PUBLIC_PER_EPOCH` (1,500) drawn each
epoch, not all of them. The number of epochs is set from a speed measurement (`EPOCHS = None`
means auto, from `TIME_BUDGET_H` hours per fold) and is much smaller than their 50 with
patience 10. The cosine schedule always runs to its end and the best validation epoch is kept,
because nb21 showed that early stopping on a schedule that has not decayed stops training at
peak learning rate. Warmup is 3 epochs. Per-task loss weights are fixed, not learned.
Validation each epoch averages 3 conformers; the final predictions use `K_EVAL` (10) conformers.

**Folds and comparability.** The five development folds are nb5's `GroupKFold(5)` on scaffold
groups, so out-of-fold Spearman compares with `mtl13` (0.5577 / 0.6935 / 0.4515 / 0.8022,
macro dev 0.6262, macro holdout 0.6652). About 10% of each fold's training compounds, held out
by scaffold, select the best epoch, so each model trains on about 72% of the development rows.

**Gate.** Macro dev Spearman at least +0.015 over `mtl13` *or* a clearly lower correlation with
`n15` and `mtl13` than earlier arms (below about 0.85), then the usual blend test against `v25`.
An arm at `mtl13`'s level with different errors is worth a blend test too.

**Speed is unmeasured on a GPU.** The benchmark cell times a few training and inference
batches, prints the projected time per epoch and per fold, and (with `EPOCHS = None`) sizes the
run to `TIME_BUDGET_H`. The five folds can be split across runtimes: give each runtime its own
`RUN` letter and its own `FOLDS` list, then run the last cell anywhere once all five exist.

**Safe to run beside notebooks 19 to 24 and beside copies of itself.** It reads only the shared
payload and writes `nb25_*` files (`nb25_f<k>.npz`, `ajft_<iso>.npz`, `results_nb25_<RUN>.json`,
its own `aid1851_nb25.csv`, `chembl_nb25.csv` and `nb25_conf_*.pkl` caches), each saved under a
random temporary name then renamed. The zip holds only its own files. A disconnect loses at
most the fold in progress.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU: switch the runtime to a GPU before going on')

In [ ]:
RUN = 'a'                      # a letter per runtime when running folds in parallel
FOLDS = [0, 1, 2, 3, 4]        # which development folds this runtime trains
EPOCHS = None                  # None: set from the speed benchmark and TIME_BUDGET_H
TIME_BUDGET_H = 1.0            # hours per fold when EPOCHS is None
BATCH = 8                      # 32 ran out of memory on a 22 GB L4 (bf16, no cuEquivariance); 8 with GRAD_CKPT is the safe start
GRAD_CKPT = True               # trades speed for memory; needed at this batch size on an L4
K_CONF, K_PUB = 10, 3          # conformers generated per challenge compound / per public compound
K_EVAL = 10                    # conformers averaged in the final predictions (5 halves that cost)
PUBLIC_POOL, PUBLIC_PER_EPOCH = 6000, 1500
VAL_CONFS = 3                  # conformers averaged for the per-epoch validation
LR_ENC, HEAD_MULT, WARMUP_EPOCHS = 5e-5, 10.0, 3
AUX_WEIGHT, EMA_DECAY, L2SP = 0.3, 0.99, 1e-5
LIMIT = None                   # debugging only: use the first N compounds of each set
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # before torch touches CUDA
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
assert os.path.ismount('/content/drive'), 'Drive is not mounted; the run could not resume without it'
NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy', 'scaffold_groups.npy']
have = sorted(os.listdir('/content/cache'))
missing = [f for f in NEED if f not in have]
assert not missing, 'payload is missing %s. Found: %s' % (missing, have)
print('run', RUN, '| folds', FOLDS, '| payload ok')

In [ ]:
!pip -q install "atom-jepa[rdkit]"

In [ ]:
import gc, json, os, pickle, shutil, time, urllib.request, uuid, zipfile
import numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISO = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']; PAN = ['CYP1A2', 'CYP2C9', 'CYP2C19', 'CYP2D6', 'CYP3A4']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
meta = pd.read_csv(CACHE / 'meta_train.csv.gz'); mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy'); G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
SM_TR = meta['SMILES'].to_numpy(); SM_TE = mete['SMILES'].to_numpy()
assert len(meta) == len(hold) == len(G)

def save_pickle_atomic(path, obj):
    tmp = Path(f'{path}.{uuid.uuid4().hex}.tmp'); pickle.dump(obj, open(tmp, 'wb')); os.replace(tmp, path)
def save_npz_atomic(path, **kw):
    tmp = Path(f'{path}.{uuid.uuid4().hex}.tmp.npz'); np.savez_compressed(tmp, **kw); os.replace(tmp, path)

def cached(name, url, need_cols):
    """Download once to Drive under this notebook's own name, via a temporary file."""
    p = OUT / name
    def ok(q):
        try: return set(need_cols) <= set(pd.read_csv(q, nrows=0).columns)
        except Exception: return False
    if not (p.exists() and ok(p)):
        print('downloading', name)
        tmp = Path('/content') / (name + '.dl'); urllib.request.urlretrieve(url, str(tmp))
        assert ok(tmp), f'{name}: download lacks {need_cols}'
        part = OUT / f'{name}.{uuid.uuid4().hex}.tmp'; shutil.copyfile(tmp, part); os.replace(part, p)
    return p

AID = cached('aid1851_nb25.csv',
    'https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
    '&record_type=datatable&actvty=all&response_type=save&aid=1851',
    ['Max_Response', 'Panel Name', 'PUBCHEM_EXT_DATASOURCE_SMILES'])
CHB = cached('chembl_nb25.csv',
    'https://raw.githubusercontent.com/jeremycheminf/openadmet_scripts/main/'
    'CYP_Challenge/data/external/cyp_chembl.csv', ['smiles'] + [f'{p}_IC50_PIC50' for p in PAN])
raw = pd.read_csv(AID, low_memory=False, skiprows=[1])
raw['panel'] = raw['Panel Name'].map({'p450-cyp1a2': 'CYP1A2', 'p450-cyp2c9': 'CYP2C9', 'p450-cyp2c19': 'CYP2C19',
                                      'p450-cyp2d6': 'CYP2D6', 'p450-cyp3a4': 'CYP3A4'})
# Negated so larger means more inhibition; clipped because a few rows reach +1330.
raw['mr'] = (-pd.to_numeric(raw['Max_Response'], errors='coerce')).clip(-50, 150)
raw = raw.dropna(subset=['panel', 'mr', 'PUBCHEM_EXT_DATASOURCE_SMILES'])
chembl = pd.read_csv(CHB)
print('AID 1851 rows', len(raw), '| ChEMBL rows', len(chembl))

## The 18-target table

As in nb21: one row per compound, keyed by the InChIKey connectivity block of the largest
fragment; any public key that matches a challenge training or blind compound is dropped.

In [ ]:
def key_and_smiles(s):
    m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
    if m is None: return None, None
    m = max(Chem.GetMolFrags(m, asMols=True), key=lambda f: f.GetNumHeavyAtoms())
    k = Chem.MolToInchiKey(m)
    return (k.split('-')[0] if k else None), Chem.MolToSmiles(m)

ours = set()
for s in np.concatenate([SM_TR, SM_TE]):
    k, _ = key_and_smiles(s)
    if k: ours.add(k)
for frame in (meta, mete):
    if 'conn_key' in frame: ours |= {str(v).split('-')[0] for v in frame['conn_key'].dropna()}

def keyed(smiles):
    ks = {s: key_and_smiles(s) for s in pd.unique(smiles)}
    return [ks[s][0] for s in smiles], [ks[s][1] for s in smiles]

raw['key'], raw['smi'] = keyed(raw['PUBCHEM_EXT_DATASOURCE_SMILES'].to_numpy())
chembl['key'], chembl['smi'] = keyed(chembl['smiles'].to_numpy())
raw = raw.dropna(subset=['key']); chembl = chembl.dropna(subset=['key'])
mr = raw.pivot_table(index='key', columns='panel', values='mr', aggfunc='median').reindex(columns=PAN)
ch = chembl.groupby('key')[[f'{p}_IC50_PIC50' for p in PAN]].median(); ch.columns = PAN
pub_keys = sorted((set(mr.index) | set(ch.index)) - ours)
smi_of = dict(zip(raw['key'], raw['smi'])); smi_of.update(zip(chembl['key'], chembl['smi']))
print(f'public compounds {len(pub_keys)}')

HEADS = ([f'chal_{i}' for i in ISO] + [f'scr_{i}' for i in ISO]
         + [f'ch_{p}' for p in PAN] + [f'mr_{p}' for p in PAN])
NT = len(HEADS); n_ch = len(SM_TR); n_pub = len(pub_keys)
SM_ALL = np.concatenate([[key_and_smiles(s)[1] or s for s in SM_TR], np.array([smi_of[k] for k in pub_keys], dtype=object)])
Y = np.full((n_ch + n_pub, NT), np.nan)
for j, i in enumerate(ISO):
    Y[:n_ch, j] = meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4 + j] = meta[f'{i}_log2fc'].to_numpy(float)
Y[n_ch:, 8:13] = ch.reindex(pub_keys).to_numpy(float)
Y[n_ch:, 13:18] = mr.reindex(pub_keys).to_numpy(float)
SM_TE_F = np.array([key_and_smiles(s)[1] or s for s in SM_TE], dtype=object)
print('table', Y.shape, '| labels per head:', dict(zip(HEADS, np.isfinite(Y).sum(0).tolist())))

y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold); ho_rows = np.flatnonzero(obs_any & hold)
rng0 = np.random.default_rng(0)
pool = n_ch + rng0.choice(n_pub, min(PUBLIC_POOL, n_pub), replace=False)
if LIMIT:
    dev_rows, ho_rows = dev_rows[:LIMIT], ho_rows[:LIMIT]; pool = pool[:LIMIT]
    SM_TE_F = SM_TE_F[:LIMIT]; SM_TE = SM_TE[:LIMIT]
need_conf = np.unique(np.r_[dev_rows, ho_rows, pool])
print('compounds needing conformers: challenge', len(dev_rows) + len(ho_rows), '| public pool', len(pool),
      '| blind', len(SM_TE_F))

## Conformers

ETKDGv3 then MMFF94, hydrogens included, from `atom_jepa.conformers.smiles_to_samples` (the
protocol in the paper). Built in parallel, saved to Drive every 400 molecules, and resumed after a
disconnect. A molecule that fails is retried with fewer conformers, then dropped (public) or
reported (challenge).

In [ ]:
from multiprocessing import Pool
from atom_jepa.conformers import smiles_to_samples

def one(args):
    smi, k = args
    for kk in (k, 5, 3, 1):
        if kk > k: continue
        try:
            s = smiles_to_samples(smi, num_conformers=kk, seed=42)
            return smi, (s[0]['atomic_numbers'].numpy().astype(np.int16),
                         np.stack([x['node_coordinates'].numpy() for x in s]).astype(np.float32))
        except Exception:
            continue
    return smi, None

def build_conformers(smiles, k, tag):
    f = OUT / f'nb25_conf_{tag}.pkl'
    store = pickle.load(open(f, 'rb')) if f.exists() else {}
    todo = [s for s in dict.fromkeys(smiles) if s not in store]
    print(f'{tag}: {len(store)} cached, {len(todo)} to build (k={k})', flush=True)
    t0 = time.time()
    with Pool(max(1, (os.cpu_count() or 2))) as pool_:
        for a in range(0, len(todo), 400):
            for smi, res in pool_.imap_unordered(one, [(s, k) for s in todo[a:a + 400]]):
                store[smi] = res
            save_pickle_atomic(f, store)
            print(f'  {tag}: {min(a + 400, len(todo))}/{len(todo)} ({time.time()-t0:.0f}s)', flush=True)
    return store

C_CH = build_conformers(list(SM_ALL[np.r_[dev_rows, ho_rows]]), K_CONF, 'chal')
C_TE = build_conformers(list(SM_TE_F), K_CONF, 'blind')
C_PUB = build_conformers(list(SM_ALL[pool]), K_PUB, 'pub')

CONF = {}                                   # row index -> (atomic numbers, positions [K, N, 3])
for i in np.r_[dev_rows, ho_rows]:
    if C_CH.get(SM_ALL[i]) is not None: CONF[i] = C_CH[SM_ALL[i]]
for i in pool:
    if C_PUB.get(SM_ALL[i]) is not None: CONF[i] = C_PUB[SM_ALL[i]]
CONF_TE = [C_TE.get(s) for s in SM_TE_F]
bad_ch = [i for i in np.r_[dev_rows, ho_rows] if i not in CONF]
bad_te = [i for i, c in enumerate(CONF_TE) if c is None]
print(f'conformers ready: challenge {len(dev_rows)+len(ho_rows)-len(bad_ch)} of {len(dev_rows)+len(ho_rows)}, '
      f'public {sum(1 for i in pool if i in CONF)} of {len(pool)}, blind {len(CONF_TE)-len(bad_te)} of {len(CONF_TE)}')
print('conformers per molecule (challenge):',
      pd.Series([CONF[i][1].shape[0] for i in np.r_[dev_rows, ho_rows] if i in CONF]).value_counts().sort_index().to_dict())
assert len(bad_ch) + len(bad_te) <= 0.02 * (len(dev_rows) + len(ho_rows) + len(CONF_TE)), \
    'more than 2% of challenge molecules failed to embed; look at them before going on'
dev_rows = np.array([i for i in dev_rows if i in CONF]); ho_rows = np.array([i for i in ho_rows if i in CONF])
pool = np.array([i for i in pool if i in CONF])

## The model

`AtomJEPA.from_pretrained('molecules')` is the encoder. Per-atom l=0 scalars and per-degree
norms of the l>0 components (computed per atom before pooling, as in their code) are mean-pooled
to 768 numbers, then a shared trunk and one small head per target.

In [ ]:
import copy, math, logging
from atom_jepa import AtomJEPA
import torch.nn as nn, torch.nn.functional as F

CAP = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
AJ_KW = dict(precision='bf16' if CAP >= 8 else 'fp32', cuequivariance=False, compile=False)
if DEVICE == 'cpu': AJ_KW['precision'] = 'fp32'
print('GPU compute capability', CAP, '| encoder settings', AJ_KW)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.aj = AtomJEPA.from_pretrained('molecules', **AJ_KW, verbose=False)
        if GRAD_CKPT: self.aj.set_grad_checkpointing(True)
        C = self.aj.embedding_dim
        self.trunk = nn.Sequential(nn.Linear(3 * C, 256), nn.SiLU(), nn.Dropout(0.1))
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(256, 64), nn.SiLU(), nn.Linear(64, 1))
                                    for _ in range(NT)])
        self.to(self.aj.device)

    def forward(self, batch):
        x = self.aj(batch, degrees='all', invariant=True, per_atom=True)       # [atoms, 3, C]
        gi = batch['node_graph_index'].to(x.device); B = int(batch['num_graphs'])
        s = torch.zeros(B, *x.shape[1:], device=x.device, dtype=x.dtype).index_add_(0, gi, x)
        n = torch.bincount(gi, minlength=B).clamp(min=1).to(x.dtype)[:, None, None]
        z = self.trunk((s / n).flatten(1))
        return torch.cat([h(z) for h in self.heads], 1)

class EMA:
    """Exponential moving average of all parameters; swap() puts it in for evaluation."""
    def __init__(self, net, decay):
        self.p = [q for q in net.parameters() if q.requires_grad]; self.decay = decay
        self.shadow = [q.detach().clone() for q in self.p]; self.backup = None
    @torch.no_grad()
    def update(self): torch._foreach_lerp_(self.shadow, [q.detach() for q in self.p], 1.0 - self.decay)
    @torch.no_grad()
    def swap(self):
        if self.backup is None:
            self.backup = [q.detach().clone() for q in self.p]
            for q, s in zip(self.p, self.shadow): q.copy_(s)
        else:
            for q, b in zip(self.p, self.backup): q.copy_(b)
            self.backup = None

# standardise every head with statistics from development and public-pool rows only
_fit = np.r_[dev_rows, pool]
MU = np.nanmean(Y[_fit], 0); SD = np.where(np.nanstd(Y[_fit], 0) > 0, np.nanstd(Y[_fit], 0), 1.0)
YZ = ((Y - MU) / SD).astype(np.float32)
TW = torch.tensor(np.r_[np.ones(4), np.full(NT - 4, AUX_WEIGHT)], dtype=torch.float32)

def sample_of(z, pos, y=None):
    d = {'atomic_numbers': torch.from_numpy(z.astype(np.int64)), 'node_coordinates': torch.from_numpy(pos)}
    if y is not None: d['y'] = torch.from_numpy(y)
    return d

class TrainDS(torch.utils.data.Dataset):
    def __init__(self, rows, choice): self.rows, self.choice = rows, choice
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        z, pos = CONF[self.rows[i]]
        return sample_of(z, pos[self.choice[i] % pos.shape[0]], YZ[self.rows[i]])

class EvalDS(torch.utils.data.Dataset):
    def __init__(self, confs, k_use):
        self.items = [(a, c) for a, cf in enumerate(confs) for c in range(min(k_use, cf[1].shape[0]))]
        self.confs = confs
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        a, c = self.items[i]; z, pos = self.confs[a]
        return sample_of(z, pos[c])

NW = int(os.environ.get('AJ_WORKERS', 2 if DEVICE == 'cuda' else 0))
_probe = AtomJEPA.from_pretrained('molecules', **AJ_KW, verbose=False)
COLL = _probe.collate
def collate_fn(samples): return COLL(samples)
print('encoder parameters', sum(p.numel() for p in _probe.parameters()), '| embedding dim', _probe.embedding_dim)
del _probe

def loader(ds, bs, shuffle):
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate_fn,
                                       num_workers=NW, persistent_workers=False)

@torch.no_grad()
def predict_std(net, confs, k_use, bs=64):
    """Standardised predictions [n_compounds, NT], each the mean over its conformers."""
    net.eval(); ds = EvalDS(confs, k_use)
    acc = np.zeros((len(confs), NT)); cnt = np.zeros(len(confs)); pos = 0
    for batch in loader(ds, bs, False):
        n = int(batch['num_graphs']); p = net(batch).float().cpu().numpy()
        for r in range(n):
            a = ds.items[pos + r][0]; acc[a] += p[r]; cnt[a] += 1
        pos += n
    return acc / cnt[:, None]

def huber_masked(pred, target, weights):
    m = torch.isfinite(target).float() * weights
    l = F.huber_loss(pred, torch.nan_to_num(target), reduction='none', delta=1.0) * m
    return l.sum() / m.sum().clamp(min=1.0), m

## Speed benchmark

Times a few training steps and inference batches on this runtime, prints the projected time per
epoch and per fold, and sets `EPOCHS` when it is `None`. If the projected time per epoch is far
longer than expected, lower `PUBLIC_PER_EPOCH` or `BATCH` before the long run.

In [ ]:
net = Net(); net.train()
params = [q for q in net.parameters() if q.requires_grad]
opt = torch.optim.AdamW(params, lr=1e-6)
rows = np.r_[dev_rows[:BATCH * 6]]
ds = TrainDS(rows, np.zeros(len(rows), int)); ld = loader(ds, BATCH, True)
times = []
for it, batch in enumerate(ld):
    if it == 6: break
    t0 = time.time()
    loss, _ = huber_masked(net(batch).float(), batch['y'].to(net.aj.device), TW.to(net.aj.device))
    opt.zero_grad(); loss.backward(); opt.step()
    if DEVICE == 'cuda': torch.cuda.synchronize()
    times.append(time.time() - t0)
step_s = float(np.median(times[2:])) if len(times) > 3 else float(np.median(times))
t0 = time.time(); _ = predict_std(net, [CONF[i] for i in dev_rows[:128]], 1); inf_rate = 128 / (time.time() - t0)
n_train = int(0.9 * 0.8 * len(dev_rows)) + PUBLIC_PER_EPOCH
n_val = int(0.1 * 0.8 * len(dev_rows))
epoch_s = (n_train / BATCH) * step_s + n_val * VAL_CONFS / inf_rate
final_s = (0.2 * len(dev_rows) + len(ho_rows) + len(SM_TE_F)) * K_EVAL / inf_rate
print(f'train step {step_s:.2f}s per batch of {BATCH} | inference {inf_rate:.1f} conformers/s')
print(f'projected: {epoch_s/60:.1f} min per epoch (train {n_train} rows + validation), '
      f'{final_s/60:.1f} min of final 10-conformer predictions per fold')
if DEVICE == 'cuda': print('peak GPU memory so far %.1f GB' % (torch.cuda.max_memory_allocated() / 1e9))
if EPOCHS is None:
    EPOCHS = int(np.clip((TIME_BUDGET_H * 3600 - final_s) / epoch_s, 4, 30))
    print(f'EPOCHS set to {EPOCHS} for a {TIME_BUDGET_H} h per-fold budget')
total_h = len(FOLDS) * (EPOCHS * epoch_s + final_s) / 3600
print(f'projected for folds {FOLDS}: {total_h:.1f} h with {EPOCHS} epochs')
if EPOCHS <= 6: print('WARNING: only a few epochs fit the budget; lower PUBLIC_PER_EPOCH, or raise TIME_BUDGET_H')
if total_h > 10:
    print(f'This is a long run ({total_h:.0f} h of GPU). Ways to cut it: PUBLIC_PER_EPOCH = 0 (challenge compounds only), '
          'K_EVAL = 5, a smaller TIME_BUDGET_H, or split FOLDS across runtimes with different RUN letters.')
del net, opt; gc.collect(); torch.cuda.empty_cache()

## Five folds

Per fold: a fresh pretrained encoder, a scaffold-held 10% of the training compounds to choose the
best epoch, the full cosine schedule, then ten-conformer predictions for the fold's held-out
compounds, the holdout and the 750 blind compounds from the best-epoch EMA weights. Saved the
moment the fold finishes.

In [ ]:
def fold_file(k): return OUT / f'nb25_f{k}.npz'

for k, (tr, va) in enumerate(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows])):
    if k not in FOLDS: continue
    if fold_file(k).exists():
        print(f'fold {k}: already done, skipping'); continue
    t_fold = time.time()
    inner = dev_rows[tr]
    a, b = next(GroupShuffleSplit(1, test_size=0.10, random_state=42 + k).split(inner, inner, G[inner]))
    train_ch, val_rows, te_rows = inner[a], inner[b], dev_rows[va]
    torch.manual_seed(42 + k); rng = np.random.default_rng(42 + k)
    net = Net(); net.train()
    enc = [q for q in net.aj.parameters() if q.requires_grad]
    enc_ids = {id(q) for q in enc}
    trunk = list(net.trunk.parameters()); heads = [q for h in net.heads for q in h.parameters()]
    opt = torch.optim.AdamW([{'params': enc, 'lr': LR_ENC, 'weight_decay': 0.1},
                             {'params': trunk, 'lr': LR_ENC * HEAD_MULT, 'weight_decay': 0.5},
                             {'params': heads, 'lr': LR_ENC * HEAD_MULT, 'weight_decay': 0.1}])
    base_lrs = [g['lr'] for g in opt.param_groups]
    pre = [q.detach().clone() for q in enc]
    ema = EMA(net, EMA_DECAY)
    steps_per_epoch = math.ceil((len(train_ch) + PUBLIC_PER_EPOCH) / BATCH); total = EPOCHS * steps_per_epoch
    warm = WARMUP_EPOCHS * steps_per_epoch; step = 0
    val_confs = [CONF[i] for i in val_rows]
    yv = YZ[val_rows][:, :4]
    best = dict(loss=float('inf'), epoch=-1, state=None); curve = []
    for epoch in range(EPOCHS):
        t0 = time.time(); net.train()
        rows = np.r_[train_ch, rng.choice(pool, min(PUBLIC_PER_EPOCH, len(pool)), replace=False)]
        ds = TrainDS(rows, rng.integers(0, 10, len(rows)))
        run_loss, nb = 0.0, 0
        for batch in loader(ds, BATCH, True):
            f = (step + 1) / warm if step < warm else 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g, b0 in zip(opt.param_groups, base_lrs): g['lr'] = b0 * max(f, 1e-3)
            loss, _ = huber_masked(net(batch).float(), batch['y'].to(net.aj.device), TW.to(net.aj.device))
            if L2SP: loss = loss + L2SP * sum(((q - p0) ** 2).sum() for q, p0 in zip(enc, pre))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 100.0)
            opt.step(); ema.update(); step += 1; run_loss += float(loss); nb += 1
        ema.swap()
        pv = predict_std(net, val_confs, VAL_CONFS)[:, :4]
        vl = float(np.nanmean([F.huber_loss(torch.tensor(pv[:, j][np.isfinite(yv[:, j])]),
                                            torch.tensor(yv[:, j][np.isfinite(yv[:, j])])).item()
                               for j in range(4) if np.isfinite(yv[:, j]).any()]))
        if vl < best['loss']:
            best = dict(loss=vl, epoch=epoch, state=[s.detach().cpu().clone() for s in ema.shadow])
        ema.swap(); curve.append((epoch, run_loss / nb, vl))
        print(f'  fold {k} epoch {epoch:2d}: train loss {run_loss/nb:.4f} | val loss {vl:.4f}'
              f'{" *" if best["epoch"] == epoch else ""} | {(time.time()-t0)/60:.1f} min', flush=True)
    with torch.no_grad():
        for q, s in zip(ema.p, best['state']): q.copy_(s.to(q.device))
    oof = predict_std(net, [CONF[i] for i in te_rows], K_EVAL)[:, :4] * SD[:4] + MU[:4]
    hp = predict_std(net, [CONF[i] for i in ho_rows], K_EVAL)[:, :4] * SD[:4] + MU[:4]
    tp = np.full((len(SM_TE_F), 4), np.nan)
    ok = [i for i, c in enumerate(CONF_TE) if c is not None]
    tp[ok] = predict_std(net, [CONF_TE[i] for i in ok], K_EVAL)[:, :4] * SD[:4] + MU[:4]
    save_npz_atomic(fold_file(k), te=te_rows, oof=oof, ho=ho_rows, hp=hp, tp=tp,
                    best_epoch=best['epoch'], curve=np.array(curve), epochs=EPOCHS)
    r = [spearmanr(y_all[i][te_rows], oof[:, j], nan_policy='omit').statistic for j, i in enumerate(ISO)]
    print(f'fold {k}: best epoch {best["epoch"]} of {EPOCHS}, {(time.time()-t_fold)/60:.1f} min | dev rho '
          + ' '.join(f'{x:.3f}' for x in r), flush=True)
    del net, opt, ema, best; gc.collect(); torch.cuda.empty_cache()
print('folds done on this runtime:', [k for k in FOLDS if fold_file(k).exists()])

## Collect

Run this once all five fold files exist (they can come from different runtimes). Holdout and
blind predictions are the mean of the five fold models, each already averaged over ten conformers.
Outputs `ajft_<iso>.npz` in the same layout as every other arm.

In [ ]:
miss = [k for k in range(5) if not fold_file(k).exists()]
assert not miss, f'folds {miss} are not finished yet; run them (any runtime, same Drive) and rerun this cell'
oof_all = np.full((len(meta), 4), np.nan); HP, TP, info = [], [], []
for k in range(5):
    z = np.load(fold_file(k)); oof_all[z['te']] = z['oof']; HP.append(z['hp']); TP.append(z['tp'])
    info.append((int(z['best_epoch']), int(z['epochs'])))
hp = np.mean(HP, 0); tp = np.nanmean(TP, 0)
ho_rows = np.load(fold_file(0))['ho']
print('best epoch / epochs per fold:', info)
RESULTS = []
for j, i in enumerate(ISO):
    y = y_all[i]; d = dev_rows[np.isfinite(y[dev_rows])]; h = ho_rows[np.isfinite(y[ho_rows])]
    hsel = np.isin(ho_rows, h)
    save_npz_atomic(OUT / f'ajft_{i}.npz', oof=oof_all[d, j], dev=d, hold=hp[hsel, j], ho=h, test=tp[:, j])
    r = float(spearmanr(y[d], oof_all[d, j]).statistic); rh = float(spearmanr(y[h], hp[hsel, j]).statistic)
    RESULTS.append(dict(iso=i, dev_spearman=r, holdout_spearman=rh))
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}')
json.dump(dict(epochs=info, results=RESULTS), open(OUT / f'results_nb25_{RUN}.json', 'w'), indent=2)
print('\nmtl13 (nb5): 0.5577 0.6935 0.4515 0.8022 | macro dev 0.6262 | macro holdout 0.6652')
print('ajft macro dev %.4f | macro holdout %.4f' % (np.mean([x['dev_spearman'] for x in RESULTS]),
                                                     np.mean([x['holdout_spearman'] for x in RESULTS])))
mine = [fold_file(k) for k in range(5)] + [OUT / f'ajft_{i}.npz' for i in ISO] + [OUT / f'results_nb25_{RUN}.json']
with zipfile.ZipFile(f'/content/nb25_{RUN}_results.zip', 'w') as zf:
    for p in mine: zf.write(p, p.name)
try:
    from google.colab import files
    files.download(f'/content/nb25_{RUN}_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- the results are on Drive in', OUT)